# LoRA fine-tune: pipeline-fault SLM

Trains a small model to read a synthetic pipeline failure log and emit a JSON remediation action. Runs top to bottom on a free Colab/Kaggle T4. All data is synthetic.

In [ ]:
!pip install -q transformers peft trl datasets accelerate

In [ ]:
import os, subprocess
if not os.path.exists("pipeline-fault-slm"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/tariniprasadtpn3/pipeline-fault-slm"], check=True)
%cd pipeline-fault-slm
!python -m src.dataset.builder --out data --n 600 --seed 42

In [ ]:
from datasets import load_dataset
ds = load_dataset("json", data_files={"train": "data/train.jsonl", "validation": "data/val.jsonl"})
ds = ds.map(lambda r: {"messages": r["messages"]}, remove_columns=[c for c in ds["train"].column_names if c != "messages"])
print(ds)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
MODEL = "Qwen/Qwen2.5-0.5B-Instruct"   # small enough for a T4
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16)

In [ ]:
from peft import LoraConfig
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])

In [ ]:
from trl import SFTConfig, SFTTrainer
cfg = SFTConfig(output_dir="out", num_train_epochs=2, per_device_train_batch_size=4,
                gradient_accumulation_steps=4, learning_rate=2e-4, logging_steps=10,
                eval_strategy="epoch", save_strategy="no", fp16=True, max_length=512,
                report_to="none")
trainer = SFTTrainer(model=model, args=cfg, train_dataset=ds["train"],
                     eval_dataset=ds["validation"], peft_config=lora, processing_class=tok)
trainer.train()
trainer.model.save_pretrained("lora-adapter"); tok.save_pretrained("lora-adapter")

In [ ]:
import json
from src.eval.harness import evaluate, load_jsonl, format_report
from src.dataset.builder import SYSTEM_PROMPT
model = trainer.model.eval()
def predict(prompt):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=96, do_sample=False)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
test = load_jsonl("data/test.jsonl")
print(format_report(evaluate(test, predict), "lora-slm"))